# 03.2 — DataLoader and collation

A `Dataset` hands out one example; training wants **batched tensors**. The `DataLoader` is the machinery in between — batching, shuffling, parallel loading — and **collation** is the specific step where a list of examples becomes one batch. It's also where this project departs from PyTorch's defaults, so the customization is worth understanding.

This notebook covers:

* The `DataLoader` loop vs MATLAB's `read`/`hasdata` cursor and `minibatchqueue`.
* Default collation — and exactly when it breaks.
* This project's `collate_trials`: dict batches with a metadata list.
* `num_workers`: parallel `.mat` loading.

**Prerequisite:** [03.1 Dataset vs fileDatastore](03.1_dataset_vs_filedatastore.ipynb).

## Section 1 — What MATLAB does

Manual batching over a datastore is a cursor loop:

```matlab
reset(DataStore);
while hasdata(DataStore)
    batch = cell(MiniBatchSize, 1);
    for b = 1:MiniBatchSize
        if hasdata(DataStore); batch{b} = read(DataStore); end
    end
    X = cat(4, batch{:});    % assemble the batch tensor BY HAND
    ...
end
```

The toolbox's `minibatchqueue` automates this — batch size, a `MiniBatchFcn` for custom assembly, `MiniBatchFormat='SSCTB'` labels, background prefetch via `DispatchInBackground`. The `cgg_trainNetwork` loop builds batches from pre-split datastore partitions ([03.3](03.3_the_session_balanced_sampler.ipynb) covers that split).

`DataLoader` = `minibatchqueue`, with `collate_fn` playing the role of `MiniBatchFcn`.

## Section 2 — The Python concepts you need

### 2.1 — The DataLoader loop

In [ ]:
import torch
from torch.utils.data import DataLoader
from neural_data_decoding.data import SyntheticTrialDataset
from neural_data_decoding.data.dataset import collate_trials

ds = SyntheticTrialDataset(
    num_sessions=2, trials_per_session=10, num_samples=10,
    num_features=4, num_classes_per_dim=[3], seed=0,
)

loader = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_trials)

for batch in loader:                       # one iteration = one batch
    print(type(batch).__name__, "with keys:", list(batch.keys()))
    print("x:      ", tuple(batch["x"].shape), "  ← batch axis prepended")
    print("targets:", tuple(batch["targets"].shape))
    print("metadata:", batch["metadata"][:2], "…")
    break
print("batches per epoch:", len(loader))    # ceil(20 / 8) = 3

What each constructor argument replaced:

| DataLoader arg | MATLAB analog | Notes |
|---|---|---|
| `batch_size=8` | `MiniBatchSize` | (unless a batch_sampler owns it — [03.3](03.3_the_session_balanced_sampler.ipynb)) |
| `shuffle=True` | `shuffle(ds)` per epoch | permutes INDICES — the dataset itself never moves |
| `collate_fn=...` | `MiniBatchFcn` | list-of-examples → one batch (§2.2) |
| `num_workers=N` | `DispatchInBackground` | N subprocesses each running `__getitem__` (§2.4) |
| `drop_last=True` | `PartialMiniBatch='discard'` | drop the ragged final batch |

And `hasdata`/`reset` vanish entirely: a `for batch in loader:` loop IS an epoch, and starting a new `for` loop IS the reset.

### 2.2 — Collation: list of examples → one batch

Conceptually: the loader picks indices `[3, 17, 8, …]`, calls `ds[i]` for each, and hands the **list of results** to `collate_fn`. The default (`torch.utils.data.default_collate`) stacks tensors along a new dim 0 and recurses into tuples/dicts — fine for uniform tensor tuples:

In [ ]:
from torch.utils.data import default_collate

examples = [(torch.randn(5), torch.tensor(1)), (torch.randn(5), torch.tensor(0))]
xb, yb = default_collate(examples)
print(xb.shape, yb.shape)      # (2, 5) and (2,) — stacked along new dim 0

But our examples are `(x, targets, metadata)` where metadata is a **dict of mixed types** (`session_id`, `trial_id`, and for real data a `session_name` *string*). Default collation tries to be clever with dicts — it collates each key, turning N metadata dicts into one dict of stacked values, and stumbles on strings and on anything ragged. When defaults get clever, pipelines get fragile — so this project writes the ten-line custom function instead:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/dataset.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def collate_trials"))
for k in range(i, min(i + 24, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* Tensors get `torch.stack` (batch axis prepended — the `(B, W, T, A, C)` from [02.2](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb) is born here).
* Metadata stays a **plain list of dicts** — no stacking, no type gymnastics. Downstream code that wants per-trial info indexes the list.
* The return is a dict, so training-loop code reads `batch["x"]` — self-documenting at every call site.

### 2.3 — Batch-shape bookkeeping, end to end

The full shape story in one cell — where every axis comes from:

In [ ]:
x_single, targets_single, _ = ds[0]
batch = next(iter(DataLoader(ds, batch_size=4, collate_fn=collate_trials)))
print(f"one example:  x {tuple(x_single.shape)}   targets {tuple(targets_single.shape)}")
print(f"one batch:    x {tuple(batch['x'].shape)}  targets {tuple(batch['targets'].shape)}")
print("the DataLoader added exactly one axis, at position 0 — nothing else changed")

### 2.4 — `num_workers`: parallel loading

With `num_workers=N`, the loader spawns N subprocesses, each running `__getitem__` calls and shipping finished examples back — so `.mat` parsing overlaps with GPU compute. For `MatFileTrialDataset` (one file-read per example) this is the difference between the GPU waiting on disk and not.

Practical rules:

* `num_workers=0` (default) = load in the main process. **Keep it 0 in notebooks** — worker subprocesses on macOS/Windows use spawn, which re-imports the calling module and breaks on notebook-defined datasets.
* In training scripts, 2–8 workers is typical; this project's CLI keeps the default and relies on lazy loading, which is fine at current trial sizes.
* Each worker gets its own RNG state — relevant for augmentation ([03.6](03.6_augmentation_per_call_contract.ipynb)).

## Section 3 — The `neural_data_decoding` implementation

Where the loaders are built in the CLI — synthetic and real data flowing through the SAME construction (the payoff of the shared protocol from 03.1):

In [ ]:
src = Path("../../src/neural_data_decoding/cli.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "train_loader = DataLoader(" in src[j])
for k in range(i, min(i + 7, len(src))):
    print(f"{k + 1:4} | {src[k]}")

One nuance you can now read critically: this call uses `shuffle=True` + `batch_size`. The MATLAB pipeline's batching is *session-constrained* — and the Python twin of that constraint is the `SingleSessionBatchSampler`, which replaces both arguments. That's the next notebook.

## Section 4 — Hands-on exercises

### Exercise 4.1 — break default collation, then fix it

Build a 3-example list where each example is `(tensor, {"name": str})` and watch `default_collate` mangle it; then write the 4-line collate that returns `(stacked_tensor, list_of_dicts)`.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
examples = [(torch.randn(3), {"name": f"trial_{i}"}) for i in range(3)]

mangled = default_collate(examples)
print("default_collate output:", type(mangled[1]), mangled[1])   # dict of LISTS — name-major, not trial-major

def collate_keep_meta(batch):
    xs = torch.stack([b[0] for b in batch])
    metas = [b[1] for b in batch]            # leave dicts alone
    return xs, metas

xs, metas = collate_keep_meta(examples)
print("custom output:        ", xs.shape, metas)

(Note what default_collate did: it inverted the structure — one dict holding a list per key — rather than failing. Silent structure inversion is worse than an error, which is exactly why `collate_trials` exists.)

### Exercise 4.2 — epoch arithmetic

`len(ds) == 20`, `batch_size=8`. How many batches with `drop_last=False` vs `True`? What are the batch shapes? Verify both.

In [ ]:
# Reveal:
for drop in (False, True):
    dl = DataLoader(ds, batch_size=8, drop_last=drop, collate_fn=collate_trials)
    shapes = [tuple(b["x"].shape) for b in dl]
    print(f"drop_last={drop}:  {len(dl)} batches, shapes {shapes}")

## Section 5 — Common errors

### `TypeError: default_collate: batch must contain tensors ... found <class 'str'>`

String (or other non-numeric) fields hit the default collator. Custom `collate_fn` that keeps metadata as a list — §2.2.

### `RuntimeError: stack expects each tensor to be equal size`

Ragged examples — trials with different window counts, sequences of different lengths. Either standardize in the dataset (this project's windowing produces fixed `(W, T, A, C)` per config) or write a padding collate.

### Notebook hangs / crashes with `num_workers > 0`

The spawn problem (§2.4). In notebooks keep `num_workers=0`; in scripts, guard the entry point with `if __name__ == "__main__":`.

### `ValueError: batch_size option ... mutually exclusive with batch_sampler`

You passed both a `batch_sampler` and `batch_size`/`shuffle`/`drop_last`. A batch sampler owns ALL batching decisions — the loader takes it or the scalar options, never both ([03.3](03.3_the_session_balanced_sampler.ipynb)).

### Every epoch has identical batch order despite `shuffle=True`

Someone called `torch.manual_seed(...)` inside the loop, re-seeding the generator each epoch. Seed once, up front — or hand the loader its own `generator=`.

### First batch is slow, rest are fast

Normal: worker startup + first file reads + (on GPU) CUDA context creation all land on iteration one. Benchmark from the second batch.

## Section 6 — Further reading

- [DataLoader docs](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) — every argument, including samplers, pinned memory, prefetch.
- [minibatchqueue docs](https://www.mathworks.com/help/deeplearning/ref/minibatchqueue.html) — the MATLAB counterpart, for the side-by-side.
- [Multiprocessing best practices](https://pytorch.org/docs/stable/notes/multiprocessing.html) — the full num_workers story.

Next up: **[03.3 the session-balanced sampler](03.3_the_session_balanced_sampler.ipynb)** — why a batch must never mix sessions, and the sampler that guarantees it.